In [4]:
import os

from dotenv import load_dotenv
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import (
    AnalysisResult,
    DocumentContent,
)
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from IPython.display import display, Markdown, Latex

load_dotenv()

True

In [5]:
endpoint = os.environ["AZURE_AI_ENDPOINT"]
key = os.getenv("AZURE_AI_API_KEY")
credential = AzureKeyCredential(key) if key else DefaultAzureCredential()

client = ContentUnderstandingClient(endpoint=endpoint, credential=credential)

# [START analyze_document_from_binary]
# Replace with the path to your local document file.
file_path = "sample_files/sample_invoice.pdf"

with open(file_path, "rb") as f:
    file_bytes = f.read()

print(f"Analyzing {file_path} with prebuilt-documentSearch...")

poller = client.begin_analyze_binary(
    analyzer_id="prebuilt-documentSearch",
    binary_input=file_bytes,
)
result: AnalysisResult = poller.result()
# [END analyze_document_from_binary]

# [START extract_markdown]
print("\nMarkdown Content:")
print("=" * 50)

# A PDF file has only one content element even if it contains multiple pages
content = result.contents[0]
display(Markdown(content.markdown))
print("=" * 50)

Analyzing sample_files/sample_invoice.pdf with prebuilt-documentSearch...

Markdown Content:


CONTOSO LTD.


# INVOICE

Contoso Headquarters
123 456th St
New York, NY, 10001

INVOICE: INV-100

INVOICE DATE: 11/15/2019

DUE DATE: 12/15/2019

CUSTOMER NAME: MICROSOFT CORPORATION

SERVICE PERIOD: 10/14/2019 - 11/14/2019

CUSTOMER ID: CID-12345

Microsoft Corp
123 Other St,
Redmond WA, 98052

BILL TO:
Microsoft Finance
123 Bill St,
Redmond WA, 98052

SHIP TO:
Microsoft Delivery
123 Ship St,
Redmond WA, 98052

SERVICE ADDRESS:
Microsoft Services
123 Service St,
Redmond WA, 98052


<table>
<tr>
<th>SALESPERSON</th>
<th>P.O. NUMBER</th>
<th>REQUISITIONER</th>
<th>SHIPPED VIA</th>
<th>F.O.B. POINT</th>
<th>TERMS</th>
</tr>
<tr>
<td></td>
<td>PO-3333</td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
</table>


<table>
<tr>
<th>DATE</th>
<th>ITEM CODE</th>
<th>DESCRIPTION</th>
<th>QTY</th>
<th>UM</th>
<th>PRICE</th>
<th>TAX</th>
<th>AMOUNT</th>
</tr>
<tr>
<td>3/4/2021</td>
<td>A123</td>
<td>Consulting Services</td>
<td>2</td>
<td>hours</td>
<td>$30.00</td>
<td>$6.00</td>
<td>$60.00</td>
</tr>
<tr>
<td>3/5/2021</td>
<td>B456</td>
<td>Document Fee</td>
<td>3</td>
<td></td>
<td>$10.00</td>
<td>$3.00</td>
<td>$30.00</td>
</tr>
<tr>
<td>3/6/2021</td>
<td>C789</td>
<td>Printing Fee</td>
<td>10</td>
<td>pages</td>
<td>$1.00</td>
<td>$1.00</td>
<td>$10.00</td>
</tr>
</table>


<table>
<tr>
<td>SUBTOTAL</td>
<td>$100.00</td>
</tr>
<tr>
<td>SALES TAX</td>
<td>$10.00</td>
</tr>
<tr>
<td>TOTAL</td>
<td>$110.00</td>
</tr>
<tr>
<td>PREVIOUS UNPAID BALANCE</td>
<td>$500.00</td>
</tr>
<tr>
<td>AMOUNT DUE</td>
<td>$610.00</td>
</tr>
</table>


THANK YOU FOR YOUR BUSINESS!

REMIT TO:
Contoso Billing
123 Remit St
New York, NY, 10001


In [8]:
# [START access_document_properties]
# Check if this is document content to access document-specific properties
if isinstance(content, DocumentContent):
    print(f"\nDocument type: {content.mime_type or '(unknown)'}")
    print(f"Start page: {content.start_page_number}")
    print(f"End page: {content.end_page_number}")

    # Check for pages
    if content.pages and len(content.pages) > 0:
        print(f"\nNumber of pages: {len(content.pages)}")
        for page in content.pages:
            unit = content.unit or "units"
            print(f"  Page {page.page_number}: {page.width} x {page.height} {unit}")

    # Check for tables
    if content.tables and len(content.tables) > 0:
        print(f"\nNumber of tables: {len(content.tables)}")
        table_counter = 1
        for table in content.tables:
            print(
                f"  Table {table_counter}: {table.row_count} rows x {table.column_count} columns"
            )
            table_counter += 1
# [END access_document_properties]


Document type: application/pdf
Start page: 1
End page: 1

Number of pages: 1
  Page 1: 8.5 x 11.0 LengthUnit.INCH

Number of tables: 3
  Table 1: 2 rows x 6 columns
  Table 2: 4 rows x 8 columns
  Table 3: 5 rows x 2 columns
